<a href="https://colab.research.google.com/github/atandaolaniyio/my-website/blob/main/projectPhase1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install mlxtend
!pip install scikit-learn

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
from torch.utils.data import DataLoader, random_split
from torch.utils.data.sampler import WeightedRandomSampler
import zipfile
import os
import seaborn as sns

In [ ]:
# use gpu

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [ ]:
# transform

transform = transforms.Compose([
    transforms.Resize((32, 32)),  # resize if needed
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # normalize
])

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# extract dataset and load

zip_file_path = '/content/drive/MyDrive/its365/CIFAR-10-images-master.zip'
extracted_dir = '/mnt/data/extracted_cifar10'

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extracted_dir)

dataset_dir = os.path.join(extracted_dir, 'CIFAR-10-images-master')
data_dir = os.path.join(dataset_dir, 'train')
full_dataset = torchvision.datasets.ImageFolder(os.path.join(dataset_dir, 'train'), transform=transform)

In [ ]:
# split 80/20
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# create dataloaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

# handle weight imbalance
class_count = np.bincount([y for _, y in train_dataset])
weights = 1. / class_count  # Inverse of class frequency
sample_weights = np.array([weights[y] for _, y in train_dataset])
sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

# update train with new handle
train_loader = DataLoader(train_dataset, batch_size=64, sampler=sampler)

In [ ]:
# define CNN model
class ProjectCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Dropout2d(0.1),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Dropout2d(0.2),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Dropout2d(0.3),
            nn.MaxPool2d(2)
        )

        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 10)  # CIFAR-10 has 10 classes
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.pool(x)
        x = x.view(x.size(0), -1)  # Flatten the output of convolutional layers
        return self.fc(x)

In [ ]:
model = ProjectCNN().to(device) # init model
criterion = nn.CrossEntropyLoss()  # set CEL
optimizer = optim.Adam(model.parameters(), lr=0.001) # optimize

In [ ]:
# define dataset and loader for later
test_dir = os.path.join(dataset_dir, 'test')
test_dataset = torchvision.datasets.ImageFolder(test_dir, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [ ]:
# training loop
num_epochs = 50
train_losses = []
val_losses = []

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    train_loss = running_loss / len(train_loader.dataset)
    train_losses.append(train_loss)


In [ ]:
  # validate
  model.eval()
  val_loss = 0.0
  with torch.no_grad():
    for images, labels in val_loader:
      images, labels = images.to(device), labels.to(device)

      outputs = model(images)
      loss = criterion(outputs, labels)
      val_loss += loss.item() * images.size(0)

      val_loss /= len(val_loader.dataset)
      val_losses.append(val_loss)

print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

In [ ]:
# plot train vs validation Loss
plt.plot(train_losses)
plt.plot(val_losses)
plt.title("Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend(["Train", "Validation"])
plt.show()

# classification report and confusion matrix
model.eval()
val_preds = []
val_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        val_preds.extend(predicted.cpu().numpy())
        val_labels.extend(labels.cpu().numpy())

print("Validation Set Evaluation:")
print(classification_report(val_labels, val_preds))
print("Validation Confusion Matrix:")
print(confusion_matrix(val_labels, val_preds))


In [ ]:
# test set eval
model.eval()

test_preds = []
test_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        test_preds.extend(predicted.cpu().numpy())
        test_labels.extend(labels.cpu().numpy())

In [ ]:
# print test classification and confusion matrix
print("Test Set Evaluation:")
print(classification_report(test_labels, test_preds))
print("Test Confusion Matrix:")
print(confusion_matrix(test_labels, test_preds))

In [ ]:
# plot confusion matrix to heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(confusion_matrix(test_labels, test_preds), annot=True, fmt='d', cmap='Blues', xticklabels=test_dataset.classes, yticklabels=test_dataset.classes)
plt.title("Test Set Confusion Matrix")
plt.xlabel("Predicted Labels")
plt.ylabel("True Labels")
plt.show()